In [ ]:
import pandas as pd
import numpy as np

# data paths
DATA_DIR = '/home/user2/jzhang_data/data/'
SMRI_FILE = DATA_DIR + 'ukb_brain_mri_data.csv'

#---------------------------------------#
#--------------load sMRI----------------#
#---------------------------------------#
# sMRI field IDs to extract
smri_field_ids = [
    25888, 25889, 25822, 25823, 25892, 25880, 25881, 25864, 25865, 25838,
    25839, 25840, 25841, 25900, 25902, 25901, 25903, 25905, 25904, 25844,
    25845, 25830, 25831, 25862, 25863, 25846, 25847, 25782, 25783, 25870,
    25871, 25886, 25887, 25893, 25894, 25915, 25917, 25916, 25792, 25793,
    25790, 25791, 25808, 25809, 25810, 25811, 25812, 25813, 25784, 25785,
    25828, 25829, 25832, 25833, 25826, 25827, 25824, 25825, 25852, 25853,
    25788, 25789, 25802, 25803, 25804, 25805, 25806, 25807, 25860, 25861,
    25876, 25877, 25884, 25885, 25836, 25837, 25848, 25849, 25850, 25851,
    25866, 25867, 25868, 25869, 25872, 25873, 25814, 25815, 25794, 25795,
    25842, 25843, 25882, 25883, 25834, 25835, 25786, 25787, 25816, 25817,
    25798, 25799, 25800, 25801, 25874, 25875, 25818, 25819, 25820, 25821,
    25854, 25855, 25856, 25857, 25858, 25859, 25796, 25797, 25878, 25879,
    25895, 25896, 25897, 25899, 25898, 25909, 25911, 25910, 25912, 25914,
    25913, 25906, 25908, 25907, 25890, 25891, 25918, 25919, 25920
]

# read the full MRI table
ukbb_full = pd.read_csv(SMRI_FILE)

# build the column list to keep (eid + every instance of each sMRI field)
cols_to_keep = ['eid']

for field_id in smri_field_ids:
    # find every column for this field_id (instances -0.0, -2.0, ...)
    matching_cols = [col for col in ukbb_full.columns if col.startswith(f'{field_id}-')]
    cols_to_keep.extend(matching_cols)

# keep only eid and the sMRI columns
ukbb_sMRI = ukbb_full[cols_to_keep].copy()

# drop the third imaging visit (-3.0)
ukbb_sMRI = ukbb_sMRI.loc[:, ~ukbb_sMRI.columns.str.contains('-3.0')]

# set eid as the index
ukbb_sMRI.index = pd.Index(ukbb_sMRI['eid'])
ukbb_sMRI = ukbb_sMRI.drop(columns='eid')

# drop rows with missing values
ukbb_sMRI = ukbb_sMRI.dropna()

#---------------------------------------#
#-----------brain region label----------#
#---------------------------------------#
# region label lookup
brain_regions = {
    25888: "Amygdala (left)",
    25889: "Amygdala (right)",
    25822: "Angular Gyrus (left)",
    25823: "Angular Gyrus (right)",
    25892: "Brain-Stem",
    25880: "Caudate (left)",
    25881: "Caudate (right)",
    25864: "Central Opercular Cortex (left)",
    25865: "Central Opercular Cortex (right)",
    25838: "Cingulate Gyrus, anterior division (left)",
    25839: "Cingulate Gyrus, anterior division (right)",
    25840: "Cingulate Gyrus, posterior division (left)",
    25841: "Cingulate Gyrus, posterior division (right)",
    25900: "Crus I Cerebellum (left)",
    25902: "Crus I Cerebellum (right)",
    25901: "Crus I Cerebellum (vermis)",
    25903: "Crus II Cerebellum (left)",
    25905: "Crus II Cerebellum (right)",
    25904: "Crus II Cerebellum (vermis)",
    25844: "Cuneal Cortex (left)",
    25845: "Cuneal Cortex (right)",
    25830: "Frontal Medial Cortex (left)",
    25831: "Frontal Medial Cortex (right)",
    25862: "Frontal Operculum Cortex (left)",
    25863: "Frontal Operculum Cortex (right)",
    25846: "Frontal Orbital Cortex (left)",
    25847: "Frontal Orbital Cortex (right)",
    25782: "Frontal Pole (left)",
    25783: "Frontal Pole (right)",
    25870: "Heschl's Gyrus (includes H1 and H2) (left)",
    25871: "Heschl's Gyrus (includes H1 and H2) (right)",
    25886: "Hippocampus (left)",
    25887: "Hippocampus (right)",
    25893: "I-IV Cerebellum (left)",
    25894: "I-IV Cerebellum (right)",
    25915: "IX Cerebellum (left)",
    25917: "IX Cerebellum (right)",
    25916: "IX Cerebellum (vermis)",
    25792: "Inferior Frontal Gyrus, pars opercularis (left)",
    25793: "Inferior Frontal Gyrus, pars opercularis (right)",
    25790: "Inferior Frontal Gyrus, pars triangularis (left)",
    25791: "Inferior Frontal Gyrus, pars triangularis (right)",
    25808: "Inferior Temporal Gyrus, anterior division (left)",
    25809: "Inferior Temporal Gyrus, anterior division (right)",
    25810: "Inferior Temporal Gyrus, posterior division (left)",
    25811: "Inferior Temporal Gyrus, posterior division (right)",
    25812: "Inferior Temporal Gyrus, temporooccipital part (left)",
    25813: "Inferior Temporal Gyrus, temporooccipital part (right)",
    25784: "Insular Cortex (left)",
    25785: "Insular Cortex (right)",
    25828: "Intracalcarine Cortex (left)",
    25829: "Intracalcarine Cortex (right)",
    25832: "Juxtapositional Lobule Cortex (formerly Supplementary Motor Cortex) (left)",
    25833: "Juxtapositional Lobule Cortex (formerly Supplementary Motor Cortex) (right)",
    25826: "Lateral Occipital Cortex, inferior division (left)",
    25827: "Lateral Occipital Cortex, inferior division (right)",
    25824: "Lateral Occipital Cortex, superior division (left)",
    25825: "Lateral Occipital Cortex, superior division (right)",
    25852: "Lingual Gyrus (left)",
    25853: "Lingual Gyrus (right)",
    25788: "Middle Frontal Gyrus (left)",
    25789: "Middle Frontal Gyrus (right)",
    25802: "Middle Temporal Gyrus, anterior division (left)",
    25803: "Middle Temporal Gyrus, anterior division (right)",
    25804: "Middle Temporal Gyrus, posterior division (left)",
    25805: "Middle Temporal Gyrus, posterior division (right)",
    25806: "Middle Temporal Gyrus, temporooccipital part (left)",
    25807: "Middle Temporal Gyrus, temporooccipital part (right)",
    25860: "Occipital Fusiform Gyrus (left)",
    25861: "Occipital Fusiform Gyrus (right)",
    25876: "Occipital Pole (left)",
    25877: "Occipital Pole (right)",
    25884: "Pallidum (left)",
    25885: "Pallidum (right)",
    25836: "Paracingulate Gyrus (left)",
    25837: "Paracingulate Gyrus (right)",
    25848: "Parahippocampal Gyrus, anterior division (left)",
    25849: "Parahippocampal Gyrus, anterior division (right)",
    25850: "Parahippocampal Gyrus, posterior division (left)",
    25851: "Parahippocampal Gyrus, posterior division (right)",
    25866: "Parietal Operculum Cortex (left)",
    25867: "Parietal Operculum Cortex (right)",
    25868: "Planum Polare (left)",
    25869: "Planum Polare (right)",
    25872: "Planum Temporale (left)",
    25873: "Planum Temporale (right)",
    25814: "Postcentral Gyrus (left)",
    25815: "Postcentral Gyrus (right)",
    25794: "Precentral Gyrus (left)",
    25795: "Precentral Gyrus (right)",
    25842: "Precuneous Cortex (left)",
    25843: "Precuneous Cortex (right)",
    25882: "Putamen (left)",
    25883: "Putamen (right)",
    25834: "Subcallosal Cortex (left)",
    25835: "Subcallosal Cortex (right)",
    25786: "Superior Frontal Gyrus (left)",
    25787: "Superior Frontal Gyrus (right)",
    25816: "Superior Parietal Lobule (left)",
    25817: "Superior Parietal Lobule (right)",
    25798: "Superior Temporal Gyrus, anterior division (left)",
    25799: "Superior Temporal Gyrus, anterior division (right)",
    25800: "Superior Temporal Gyrus, posterior division (left)",
    25801: "Superior Temporal Gyrus, posterior division (right)",
    25874: "Supracalcarine Cortex (left)",
    25875: "Supracalcarine Cortex (right)",
    25818: "Supramarginal Gyrus, anterior division (left)",
    25819: "Supramarginal Gyrus, anterior division (right)",
    25820: "Supramarginal Gyrus, posterior division (left)",
    25821: "Supramarginal Gyrus, posterior division (right)",
    25854: "Temporal Fusiform Cortex, anterior division (left)",
    25855: "Temporal Fusiform Cortex, anterior division (right)",
    25856: "Temporal Fusiform Cortex, posterior division (left)",
    25857: "Temporal Fusiform Cortex, posterior division (right)",
    25858: "Temporal Occipital Fusiform Cortex (left)",
    25859: "Temporal Occipital Fusiform Cortex (right)",
    25796: "Temporal Pole (left)",
    25797: "Temporal Pole (right)",
    25878: "Thalamus (left)",
    25879: "Thalamus (right)",
    25895: "V Cerebellum (left)",
    25896: "V Cerebellum (right)",
    25897: "VI Cerebellum (left)",
    25899: "VI Cerebellum (right)",
    25898: "VI Cerebellum (vermis)",
    25909: "VIIIa Cerebellum (left)",
    25911: "VIIIa Cerebellum (right)",
    25910: "VIIIa Cerebellum (vermis)",
    25912: "VIIIb Cerebellum (left)",
    25914: "VIIIb Cerebellum (right)",
    25913: "VIIIb Cerebellum (vermis)",
    25906: "VIIb Cerebellum (left)",
    25908: "VIIb Cerebellum (right)",
    25907: "VIIb Cerebellum (vermis)",
    25890: "Ventral Striatum (left)",
    25891: "Ventral Striatum (right)",
    25918: "X Cerebellum (left)",
    25919: "X Cerebellum (vermis)",
    25920: "X Cerebellum (right)"
}

# build the DataFrame
descr_dict = pd.DataFrame(list(brain_regions.items()), columns=['id', 'name'])
descr_dict = descr_dict.sort_values('id').reset_index(drop=True)

# region name list
sMRI_name = descr_dict['name'].to_list()

# summary
print(f"sMRI data shape: {ukbb_sMRI.shape}")
print(f"number of regions: {len(sMRI_name)}")
print(f"columns extracted: {len(cols_to_keep)-1}")  # minus eid
print(f"\nfirst 10 columns extracted:")
print(cols_to_keep[1:11])

In [ ]:
ukbb_sMRI.to_csv(r"/home/user2/jzhang_data/data/ukbb_sMRI_data.csv", index=True)

In [ ]:
#---------------------------------------#
#--------------load dMRI----------------#
#---------------------------------------#
# dMRI field IDs to extract
dmri_field_ids = [
    25079, 25078, 25073, 25072, 25059, 25071, 25070, 25091, 25090, 25093,
    25092, 25063, 25062, 25089, 25088, 25095, 25094, 25061, 25058, 25067,
    25066, 25065, 25064, 25056, 25057, 25083, 25082, 25075, 25074, 25085,
    25084, 25077, 25076, 25087, 25086, 25060, 25069, 25068, 25081, 25080,
    25099, 25098, 25097, 25096, 25103, 25102, 25101, 25100
]

# read the full MRI table (same file as sMRI)
ukbb_full = pd.read_csv(SMRI_FILE)

# build the column list to keep (eid + every instance of each dMRI field)
dmri_cols_to_keep = ['eid']

for field_id in dmri_field_ids:
    # find every column for this field_id
    matching_cols = [col for col in ukbb_full.columns if col.startswith(f'{field_id}-')]
    dmri_cols_to_keep.extend(matching_cols)

# keep only eid and the dMRI columns
ukbb_dMRI = ukbb_full[dmri_cols_to_keep].copy()

# drop the third imaging visit (-3.0)
ukbb_dMRI = ukbb_dMRI.loc[:, ~ukbb_dMRI.columns.str.contains('-3.0')]

# set eid as the index
ukbb_dMRI.index = ukbb_dMRI['eid']
ukbb_dMRI = ukbb_dMRI.drop(columns='eid')

# drop rows with missing values
ukbb_dMRI = ukbb_dMRI.dropna()

#---------------------------------------#
#-----------white matter tract label----#
#---------------------------------------#
# white-matter tract label lookup
white_matter_tracts = {
    25079: "anterior corona radiata (left)",
    25078: "anterior corona radiata (right)",
    25073: "anterior limb of internal capsule (left)",
    25072: "anterior limb of internal capsule (right)",
    25059: "body of corpus callosum",
    25071: "cerebral peduncle (left)",
    25070: "cerebral peduncle (right)",
    25091: "cingulum cingulate gyrus (left)",
    25090: "cingulum cingulate gyrus (right)",
    25093: "cingulum hippocampus (left)",
    25092: "cingulum hippocampus (right)",
    25063: "corticospinal tract (left)",
    25062: "corticospinal tract (right)",
    25089: "external capsule (left)",
    25088: "external capsule (right)",
    25095: "fornix cres+stria terminalis (left)",
    25094: "fornix cres+stria terminalis (right)",
    25061: "fornix",
    25058: "genu of corpus callosum",
    25067: "inferior cerebellar peduncle (left)",
    25066: "inferior cerebellar peduncle (right)",
    25065: "medial lemniscus (left)",
    25064: "medial lemniscus (right)",
    25056: "middle cerebellar peduncle",
    25057: "pontine crossing tract",
    25083: "posterior corona radiata (left)",
    25082: "posterior corona radiata (right)",
    25075: "posterior limb of internal capsule (left)",
    25074: "posterior limb of internal capsule (right)",
    25085: "posterior thalamic radiation (left)",
    25084: "posterior thalamic radiation (right)",
    25077: "retrolenticular part of internal capsule (left)",
    25076: "retrolenticular part of internal capsule (right)",
    25087: "sagittal stratum (left)",
    25086: "sagittal stratum (right)",
    25060: "splenium of corpus callosum",
    25069: "superior cerebellar peduncle (left)",
    25068: "superior cerebellar peduncle (right)",
    25081: "superior corona radiata (left)",
    25080: "superior corona radiata (right)",
    25099: "superior fronto-occipital fasciculus (left)",
    25098: "superior fronto-occipital fasciculus (right)",
    25097: "superior longitudinal fasciculus (left)",
    25096: "superior longitudinal fasciculus (right)",
    25103: "tapetum (left)",
    25102: "tapetum (right)",
    25101: "uncinate fasciculus (left)",
    25100: "uncinate fasciculus (right)"
}

# build the DataFrame
dMRI_descr_dict = pd.DataFrame(list(white_matter_tracts.items()), columns=['id', 'name'])
dMRI_descr_dict = dMRI_descr_dict.sort_values('id').reset_index(drop=True)

# tract name list
dMRI_name = dMRI_descr_dict['name'].to_list()

# summary
print(f"\ndMRI data shape: {ukbb_dMRI.shape}")
print(f"number of tracts: {len(dMRI_name)}")
print(f"dMRI columns extracted: {len(dmri_cols_to_keep)-1}")
print(f"\nfirst 5 tracts: {dMRI_name[:5]}")

In [ ]:
ukbb_dMRI.to_csv(r"/home/user2/jzhang_data/data/ukbb_dMRI_data.csv", index=True)

In [ ]:
#---------------------------------------#
#--------------load rfMRI----------------#
#---------------------------------------#
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import sys

# force flush on print
def print_flush(msg):
    print(msg)
    sys.stdout.flush()

# rfMRI data path
RFMRI_DIR = r"/home/user2/jzhang_data/data/opt/notebooks/rfMRI/"
rfmri_pattern = "*_25752_2_0.txt"

print_flush("="*60)
print_flush("Loading rfMRI data")
print_flush("="*60)

# the 21 valid ICA components defined by UK Biobank
good_components = [1, 2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]
n_networks = len(good_components)  # 21 valid components
TAR_ITEMS_PER_SUB = 210  # 21 * (21 - 1) / 2 = 210 partial correlations

print_flush(f"\nValid ICA components: {n_networks}")
print_flush(f"Expected edges: {TAR_ITEMS_PER_SUB}")

# find all rfMRI files
print_flush(f"\nSearching: {os.path.join(RFMRI_DIR, rfmri_pattern)}")
rfmri_files = glob.glob(os.path.join(RFMRI_DIR, rfmri_pattern))
print_flush(f"found {len(rfmri_files)} rfMRI files")

if len(rfmri_files) == 0:
    print_flush(f"Error: no files found")
    print_flush(f"Check the path: {RFMRI_DIR}")
    sys.exit(1)

# generate edge names
print_flush("\nGenerating edge names")
rfMRI_connection_names = []
for i in range(len(good_components)):
    for j in range(i+1, len(good_components)):
        comp_i = good_components[i]
        comp_j = good_components[j]
        rfMRI_connection_names.append(f"ICA_{comp_i}_ICA_{comp_j}")

print_flush(f"generated {len(rfMRI_connection_names)} edge names")

# check the edge count
assert len(rfMRI_connection_names) == TAR_ITEMS_PER_SUB, \
    f"edge count mismatch: expected {TAR_ITEMS_PER_SUB}, got {len(rfMRI_connection_names)}"

# read rfMRI data for every subject
print_flush("\nReading files")
rfmri_data = {}
skipped_files = []
error_files = []

# report progress every 1000 files
batch_size = 1000

for idx, file_path in enumerate(rfmri_files):
    try:
        # take eid from the filename
        filename = os.path.basename(file_path)
        eid = filename.split('_')[0]
        
        # read the file
        with open(file_path, 'r') as f:
            content = f.read().strip()
            values = [float(x) for x in content.split()]
        
        # check the value count
        if len(values) != TAR_ITEMS_PER_SUB:
            skipped_files.append((filename, len(values)))
            continue
        
        # store
        rfmri_data[eid] = values
        
        # report progress every batch_size files
        if (idx + 1) % batch_size == 0:
            print_flush(f"  processed: {idx+1}/{len(rfmri_files)} ({(idx+1)/len(rfmri_files)*100:.1f}%) | ok: {len(rfmri_data)} | skipped: {len(skipped_files)}")
        
    except Exception as e:
        error_files.append((filename, str(e)))
        if len(error_files) <= 5:  # report only the first 5 errors
            print_flush(f"  Error: {filename} - {e}")
        continue

# final progress
print_flush(f"  processed: {len(rfmri_files)}/{len(rfmri_files)} (100.0%) | ok: {len(rfmri_data)} | skipped: {len(skipped_files)}")

# convert to DataFrame
print_flush("\nBuilding DataFrame")
ukbb_rfMRI = pd.DataFrame.from_dict(rfmri_data, orient='index', columns=rfMRI_connection_names)
ukbb_rfMRI.index.name = 'eid'
print_flush(f"DataFrame built: {ukbb_rfMRI.shape}")

# cast eid to int
print_flush("\nCasting eid to int")
ukbb_rfMRI.index = ukbb_rfMRI.index.astype(int)
print_flush("eid cast to int")

# drop rows with missing values
rows_before = len(ukbb_rfMRI)
print_flush(f"\nChecking missing values")
print_flush(f"  before: {rows_before} rows")
ukbb_rfMRI = ukbb_rfMRI.dropna()
rows_after = len(ukbb_rfMRI)
print_flush(f"  after: {rows_after} rows")
if rows_before > rows_after:
    print_flush(f"  dropped {rows_before - rows_after} rows with missing values")

# detailed summary
print_flush("\n" + "="*60)
print_flush("rfMRI data loaded")
print_flush("="*60)
print_flush(f"Total files: {len(rfmri_files)}")
print_flush(f"Loaded: {len(rfmri_data)}")
print_flush(f"Skipped: {len(skipped_files)}")
print_flush(f"Errors: {len(error_files)}")

if len(skipped_files) > 0:
    print_flush(f"\nSkipped files (first 5):")
    for fname, n_vals in skipped_files[:5]:
        print_flush(f"  {fname}: {n_vals} values (expected {TAR_ITEMS_PER_SUB})")

if len(error_files) > 0:
    print_flush(f"\nError files (first 5):")
    for fname, err in error_files[:5]:
        print_flush(f"  {fname}: {err}")

print_flush(f"\nFinal data shape: {ukbb_rfMRI.shape}")
print_flush(f"  subjects: {ukbb_rfMRI.shape[0]}")
print_flush(f"  edges: {ukbb_rfMRI.shape[1]}")

# check
assert ukbb_rfMRI.shape[1] == TAR_ITEMS_PER_SUB, \
    f"feature count mismatch: expected {TAR_ITEMS_PER_SUB}, got {ukbb_rfMRI.shape[1]}"

print_flush(f"\nFirst 10 edges:")
for i in range(min(10, len(rfMRI_connection_names))):
    print_flush(f"  {i+1:3d}. {rfMRI_connection_names[i]}")


# value distribution
print_flush(f"\nValue stats:")
print_flush(f"  partial correlation range: [{ukbb_rfMRI.values.min():.4f}, {ukbb_rfMRI.values.max():.4f}]")
print_flush(f"  mean: {ukbb_rfMRI.values.mean():.4f}")
print_flush(f"  sd: {ukbb_rfMRI.values.std():.4f}")
print_flush(f"  median: {np.median(ukbb_rfMRI.values):.4f}")

# check the distribution
print_flush(f"\nPartial correlation distribution:")
n_total = ukbb_rfMRI.size
print_flush(f"  < -0.5: {(ukbb_rfMRI.values < -0.5).sum():,} ({(ukbb_rfMRI.values < -0.5).sum()/n_total*100:.2f}%)")
print_flush(f"  -0.5~0: {((ukbb_rfMRI.values >= -0.5) & (ukbb_rfMRI.values < 0)).sum():,} ({((ukbb_rfMRI.values >= -0.5) & (ukbb_rfMRI.values < 0)).sum()/n_total*100:.2f}%)")
print_flush(f"  0~0.5:  {((ukbb_rfMRI.values >= 0) & (ukbb_rfMRI.values < 0.5)).sum():,} ({((ukbb_rfMRI.values >= 0) & (ukbb_rfMRI.values < 0.5)).sum()/n_total*100:.2f}%)")
print_flush(f"  >= 0.5: {(ukbb_rfMRI.values >= 0.5).sum():,} ({(ukbb_rfMRI.values >= 0.5).sum()/n_total*100:.2f}%)")

print_flush(f"\nDone")
print_flush("="*60)

In [ ]:
ukbb_rfMRI.to_csv(r"/home/user2/jzhang_data/data/ukbb_rfMRI_data.csv", index=True)

In [ ]:
#######################################################
#          DECONFOUNDING THE BRAIN DATA               #
#######################################################
def deconf(X, subj_index):
    # deconfound the brain space once - behavior one-by-one later
    from nilearn.signal import clean
    from sklearn.preprocessing import StandardScaler
    
    sc = StandardScaler()
    X = sc.fit_transform(X)

    con_subj = subj_index.to_list()

    beh = pd.read_csv(DATA_DIR+'ukb_brain_mri_covariate.csv')
    beh.index = beh['eid']
    beh = beh.drop(columns='eid')
    beh = beh.loc[con_subj]


    # Mean rfMRI head motion
    head_motion_rest = np.nan_to_num(
        beh['25741-2.0'].values
    )
    # Mean tfMRI head motion
    head_motion_task = np.nan_to_num(
        beh['25742-2.0'].values
    )
    # Volume of grey matter
    head_size = np.nan_to_num(
        beh['25006-2.0'].values
    )

    # motivated by Elliott et al., 2018
    # exact location of the head and the radio-frequency
    # receiver coil in the scanner
    head_pos_x = np.nan_to_num(
        beh['25756-2.0'].values 
    )
    head_pos_y = np.nan_to_num(
        beh['25757-2.0'].values
    )
    head_pos_z = np.nan_to_num(
        beh['25758-2.0'].values
    )
    head_pos_table = np.nan_to_num(
        beh['25759-2.0'].values
    )
    scan_site_dummies = pd.get_dummies(
        beh['54-2.0']
    ).values

    assert not np.any(np.isnan(head_motion_rest))
    assert not np.any(np.isnan(head_motion_task))
    assert not np.any(np.isnan(head_size))

    print('Deconfounding brain feature space!')
    conf_mat = np.hstack([
        np.atleast_2d(head_motion_rest).T, np.atleast_2d(head_motion_task).T,
        np.atleast_2d(head_size).T, 
        np.atleast_2d(head_pos_x).T, np.atleast_2d(head_pos_y).T,
        np.atleast_2d(head_pos_z).T, np.atleast_2d(head_pos_table).T,
        np.atleast_2d(scan_site_dummies)
        ])

    print(len(X))
    print(len(conf_mat))
    X = clean(X, confounds=conf_mat, detrend=False, standardize=False)
    return X

In [ ]:
##### sMRI deconfound ########
import nilearn

X = deconf(ukbb_sMRI.values, ukbb_sMRI.index)
df_input_sMRI = pd.DataFrame(X, index=ukbb_sMRI.index, columns=sMRI_name)
df_input_sMRI.to_csv(DATA_DIR + 'ukbb_sMRI_deconfounded.csv', index=True)
print(f"sMRI deconfounded: {df_input_sMRI.shape}")

In [ ]:
##### dMRI  deconfound ########
X = deconf(ukbb_dMRI.values, ukbb_dMRI.index)
df_input_dMRI = pd.DataFrame(X, index=ukbb_dMRI.index, columns=dMRI_name)
df_input_dMRI.to_csv(DATA_DIR + 'ukbb_dMRI_deconfounded.csv', index=True)
print(f"dMRI deconfounded: {df_input_dMRI.shape}")

In [ ]:
##### rfMRI deconfound ########
X = deconf(ukbb_rfMRI.values, ukbb_rfMRI.index)
df_input_rfMRI = pd.DataFrame(X, index=ukbb_rfMRI.index, columns=rfMRI_connection_names)
df_input_rfMRI.to_csv(DATA_DIR + 'ukbb_rfMRI_deconfounded.csv', index=True)
print(f"rfMRI deconfounded: {df_input_rfMRI.shape}")

In [ ]:
# load phenotype data
import pandas as pd
import numpy as np
DATA_DIR = '/home/user2/jzhang_data/data/'

df_pa_obesity = pd.read_csv('/home/user2/jzhang_data/data/ukb_pa_obesity_processed.csv', index_col='eid')
print(f"Loaded: {df_pa_obesity.shape}")
print(f"\nColumns:\n{df_pa_obesity.columns.tolist()}")

In [ ]:
#----------------------------------------------------------#
#---------------define bootstrap and LDA-------------------#
#----------------------------------------------------------#
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.utils import resample
import matplotlib.pyplot as plt
import seaborn as sns

sc = StandardScaler()

# bootstrap function
def perform_bootstrap(Y, n_samples):

    df_y_test = Y.sample(frac=.1, random_state=0)
    df_y_train = Y[~Y.index.isin(df_y_test.index)]
    test_samples = df_y_test.index

    count_y_train = df_y_train.value_counts()
    minority_y = count_y_train.min()

    bootstrap_samples = []
    for _ in range(n_samples):
        # each sample have same probablity to be choiced
        np.random.seed(_)
        class0 = resample(df_y_train[df_y_train==0].dropna(), n_samples = minority_y, replace=True)
        class1 = resample(df_y_train[df_y_train==1].dropna(), n_samples = minority_y, replace=True)
        df_resmapled = pd.concat([class0, class1],axis=0) # type: ignore
        bootstrap_samples.append(df_resmapled.index.to_list())
    return bootstrap_samples, test_samples

def calculate_feature_coef(X, Y, n_BS_itr, covariates=None):
    """
    Compute LDA feature coefficients, with optional covariate adjustment.
    
    Parameters:
    -----------
    X : DataFrame
        Feature matrix
    Y : Series
        Target variable
    n_BS_itr : int
        Number of bootstrap iterations
    covariates : DataFrame, optional
        Covariate matrix; when given, X is residualised on it before LDA
    
    Returns:
    --------
    lda_estimator : ndarray
        LDA coefficient matrix (n_BS_itr x n_features)
    """
    from nilearn.signal import clean
    
    samples, test_sample = perform_bootstrap(Y, n_BS_itr)
    lda = LDA()

    lda_estimator = np.zeros((n_BS_itr, X.shape[1]))
    
    for i, sample in enumerate(samples):
        X_train = X.loc[sample].values
        y_train = Y.loc[sample].values
        y_train = np.ravel(y_train)
        
        # residualise on covariates when provided
        if covariates is not None:
            cov_train = covariates.loc[sample].values
            X_train = clean(X_train, confounds=cov_train, detrend=False, standardize=False)
        
        model = lda.fit(X_train, y_train)
        lda_estimator[i,:] = model.coef_

    return lda_estimator

def calculate_significant_coef(lda_estimator, feat_names, alpha):
    
    coef_mean = lda_estimator.mean(axis=0)
    df_coef = pd.DataFrame(coef_mean.T, index=feat_names, columns=['coef'])

    lower_bound = np.percentile(lda_estimator, alpha/2, axis=0)
    upper_bound = np.percentile(lda_estimator, 100-alpha/2, axis=0)
    e_low = coef_mean-lower_bound
    e_up = upper_bound-coef_mean

    xerr = [e_low, e_up]
    df_xerr = pd.DataFrame(np.array(xerr).T, columns=['lower','upper'], index=df_coef.index)

    bool_arry = ((lower_bound>0) | (upper_bound<0))
    df_coef_masked = df_coef.copy()
    df_coef_masked = df_coef_masked[bool_arry]
    df_xerr_masked = df_xerr.loc[df_coef_masked.index]

    df_coef_withNan = df_coef.copy()
    df_coef_withNan[~bool_arry] = np.nan

    assym_err = df_xerr_masked.values.T
    fig, ax = plt.subplots()
    ax.barh(df_coef_masked.index, df_coef_masked['coef'], xerr=assym_err)
    title = str(100-alpha/2)+'% CI coef'
    ax.set_title(title)

    # save all mean coef and CI bound
    df_bound = pd.DataFrame(np.array([lower_bound,upper_bound]).T, columns=['lower','upper'], index=df_coef.index)
    df_coef_all = pd.concat([df_coef, df_bound], axis=1)

    return df_coef_masked, df_coef_withNan, df_coef_all

In [ ]:
#----------------------------------------------------------#
#------- sMRI: HD and LD contrasts -------#
#----------------------------------------------------------#
import warnings

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

# 1. load the deconfounded sMRI matrix
df_input_sMRI = pd.read_csv(DATA_DIR + 'ukbb_sMRI_deconfounded.csv', index_col='eid')
print(f"Loaded deconfounded sMRI: {df_input_sMRI.shape}")

# 2. target variable, HD contrast
df_target_high = df_pa_obesity['high_mvpa_bmi_uncouple_2'].dropna()
common_index_high = df_target_high.index.intersection(df_input_sMRI.index)
print(f"\nHD contrast N: {len(common_index_high)}")
print(f"Label counts:\n{df_target_high.loc[common_index_high].value_counts()}")

# 3. covariates (age, sex, ...)
covariates_high = df_pa_obesity[['age', 'sex', 'education', 'sleep_duration', 'health_rating', 'illness_disability_infirmity', 'weight_changes']].loc[common_index_high]
print(f"Covariate shape: {covariates_high.shape}")

# 4. bootstrap LDA, HD contrast
print("\nRunning bootstrap LDA (HD contrast)")
n_BS_itr = 100
alpha = 10

lda_estimator_high = calculate_feature_coef(
    df_input_sMRI.loc[common_index_high], 
    df_target_high.loc[common_index_high], 
    n_BS_itr, 
    covariates=covariates_high
)

# 5. significant features
significant_coef_high, df_coef_with_nan_high, df_coef_all_high = calculate_significant_coef(
    lda_estimator_high, sMRI_name, alpha
)

# 6. save
output_file_high = DATA_DIR + 'LDA_sMRI_high_mvpa_bmi_coef_5-95CI_all.csv'
df_coef_all_high.to_csv(output_file_high)
print(f"\nHD results saved: {output_file_high}")
print(f"Significant features: {len(significant_coef_high)}")

#----------------------------------------------------------#

# 7. target variable, LD contrast
df_target_low = df_pa_obesity['low_mvpa_bmi_uncouple_2'].dropna()
common_index_low = df_target_low.index.intersection(df_input_sMRI.index)
print(f"\nLD contrast N: {len(common_index_low)}")
print(f"Label counts:\n{df_target_low.loc[common_index_low].value_counts()}")

# 8. covariates
covariates_low = df_pa_obesity[['age', 'sex', 'education', 'sleep_duration', 'health_rating', 'illness_disability_infirmity', 'weight_changes']].loc[common_index_low]

# 9. bootstrap LDA, LD contrast
print("\nRunning bootstrap LDA (LD contrast)")

lda_estimator_low = calculate_feature_coef(
    df_input_sMRI.loc[common_index_low], 
    df_target_low.loc[common_index_low], 
    n_BS_itr, 
    covariates=covariates_low
)

# 10. significant features
significant_coef_low, df_coef_with_nan_low, df_coef_all_low = calculate_significant_coef(
    lda_estimator_low, sMRI_name, alpha
)

# 11. save
output_file_low = DATA_DIR + 'LDA_sMRI_low_mvpa_bmi_coef_5-95CI_all.csv'
df_coef_all_low.to_csv(output_file_low)
print(f"\nLD results saved: {output_file_low}")
print(f"Significant features: {len(significant_coef_low)}")

print("\n" + "="*60)
print("sMRI analysis complete")
print("="*60)


In [ ]:
#----------------------------------------------------------#
#------- dMRI: HD and LD contrasts -------#
#----------------------------------------------------------#
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

# 1. load the deconfounded dMRI matrix
df_input_dMRI = pd.read_csv(DATA_DIR + 'ukbb_dMRI_deconfounded.csv', index_col='eid')
print(f"Loaded deconfounded dMRI: {df_input_dMRI.shape}")

# 2. target variable, HD contrast
df_target_high = df_pa_obesity['high_mvpa_bmi_uncouple_2'].dropna()
common_index_high = df_target_high.index.intersection(df_input_dMRI.index)
print(f"\nHD contrast N: {len(common_index_high)}")
print(f"Label counts:\n{df_target_high.loc[common_index_high].value_counts()}")

# 3. covariates (age and sex)
covariates_high = df_pa_obesity[['age', 'sex', 'education', 'sleep_duration', 'health_rating', 'illness_disability_infirmity', 'weight_changes']].loc[common_index_high]
print(f"Covariate shape: {covariates_high.shape}")

# 4. bootstrap LDA, HD contrast
print("\nRunning bootstrap LDA (HD contrast)")
n_BS_itr = 100
alpha = 10

lda_estimator_high = calculate_feature_coef(
    df_input_dMRI.loc[common_index_high], 
    df_target_high.loc[common_index_high], 
    n_BS_itr, 
    covariates=covariates_high
)

# 5. significant features
significant_coef_high, df_coef_with_nan_high, df_coef_all_high = calculate_significant_coef(
    lda_estimator_high, dMRI_name, alpha
)

# 6. save
output_file_high = DATA_DIR + 'LDA_dMRI_high_mvpa_bmi_coef_5-95CI_all.csv'
df_coef_all_high.to_csv(output_file_high)
print(f"\nHD results saved: {output_file_high}")
print(f"Significant features: {len(significant_coef_high)}")

#----------------------------------------------------------#

# 7. target variable, LD contrast
df_target_low = df_pa_obesity['low_mvpa_bmi_uncouple_2'].dropna()
common_index_low = df_target_low.index.intersection(df_input_dMRI.index)
print(f"\nLD contrast N: {len(common_index_low)}")
print(f"Label counts:\n{df_target_low.loc[common_index_low].value_counts()}")

# 8. covariates
covariates_low = df_pa_obesity[['age', 'sex', 'education', 'sleep_duration', 'health_rating', 'illness_disability_infirmity', 'weight_changes']].loc[common_index_low]

# 9. bootstrap LDA, LD contrast
print("\nRunning bootstrap LDA (LD contrast)")

lda_estimator_low = calculate_feature_coef(
    df_input_dMRI.loc[common_index_low], 
    df_target_low.loc[common_index_low], 
    n_BS_itr, 
    covariates=covariates_low
)

# 10. significant features
significant_coef_low, df_coef_with_nan_low, df_coef_all_low = calculate_significant_coef(
    lda_estimator_low, dMRI_name, alpha
)

# 11. save
output_file_low = DATA_DIR + 'LDA_dMRI_low_mvpa_bmi_coef_5-95CI_all.csv'
df_coef_all_low.to_csv(output_file_low)
print(f"\nLD results saved: {output_file_low}")
print(f"Significant features: {len(significant_coef_low)}")

print("\n" + "="*60)
print("dMRI analysis complete")
print("="*60)


In [ ]:
#----------------------------------------------------------#
#------- rfMRI: HD and LD contrasts -------#
#----------------------------------------------------------#
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

# 1. load the deconfounded rfMRI matrix
df_input_rfMRI = pd.read_csv(DATA_DIR + 'ukbb_rfMRI_deconfounded.csv', index_col='eid')
print(f"Loaded deconfounded rfMRI: {df_input_rfMRI.shape}")

# 2. target variable, HD contrast
df_target_high = df_pa_obesity['high_mvpa_bmi_uncouple_2'].dropna()
common_index_high = df_target_high.index.intersection(df_input_rfMRI.index)
print(f"\nHD contrast N: {len(common_index_high)}")
print(f"Label counts:\n{df_target_high.loc[common_index_high].value_counts()}")

# 3. covariates (age and sex)
covariates_high = df_pa_obesity[['age', 'sex', 'education', 'sleep_duration', 'health_rating', 'illness_disability_infirmity', 'weight_changes']].loc[common_index_high]
print(f"Covariate shape: {covariates_high.shape}")

# 4. bootstrap LDA, HD contrast
print("\nRunning bootstrap LDA (HD contrast)")
n_BS_itr = 100
alpha = 10

lda_estimator_high = calculate_feature_coef(
    df_input_rfMRI.loc[common_index_high], 
    df_target_high.loc[common_index_high], 
    n_BS_itr, 
    covariates=covariates_high
)

# 5. significant features
significant_coef_high, df_coef_with_nan_high, df_coef_all_high = calculate_significant_coef(
    lda_estimator_high, rfMRI_connection_names, alpha
)

# 6. save
output_file_high = DATA_DIR + 'LDA_rfMRI_high_mvpa_bmi_coef_5-95CI_all.csv'
df_coef_all_high.to_csv(output_file_high)
print(f"\nHD results saved: {output_file_high}")
print(f"Significant features: {len(significant_coef_high)}")

#----------------------------------------------------------#

# 7. target variable, LD contrast
df_target_low = df_pa_obesity['low_mvpa_bmi_uncouple_2'].dropna()
common_index_low = df_target_low.index.intersection(df_input_rfMRI.index)
print(f"\nLD contrast N: {len(common_index_low)}")
print(f"Label counts:\n{df_target_low.loc[common_index_low].value_counts()}")

# 8. covariates
covariates_low = df_pa_obesity[['age', 'sex', 'education', 'sleep_duration', 'health_rating', 'illness_disability_infirmity', 'weight_changes']].loc[common_index_low]

# 9. bootstrap LDA, LD contrast
print("\nRunning bootstrap LDA (LD contrast)")

lda_estimator_low = calculate_feature_coef(
    df_input_rfMRI.loc[common_index_low], 
    df_target_low.loc[common_index_low], 
    n_BS_itr, 
    covariates=covariates_low
)

# 10. significant features
significant_coef_low, df_coef_with_nan_low, df_coef_all_low = calculate_significant_coef(
    lda_estimator_low, rfMRI_connection_names, alpha
)

# 11. save
output_file_low = DATA_DIR + 'LDA_rfMRI_low_mvpa_bmi_coef_5-95CI_all.csv'
df_coef_all_low.to_csv(output_file_low)
print(f"\nLD results saved: {output_file_low}")
print(f"Significant features: {len(significant_coef_low)}")

print("\n" + "="*60)
print("rfMRI analysis complete")
print("="*60)
